# Uncertainty evaluation for the measurement of gauge blocks by optical interferometry
---

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
# Agrega el directorio padre (la raíz del repositorio) al path temporal de Python
sys.path.append(os.path.abspath(os.path.join('..')))

import numpy as np

from src.refraction_index import EdlenRefractiveIndex

La desviación medida entre la longitud nominal y la longitud real del bloque patrón es:

\begin{align}
    d &= l_{real} - L'\\
    d &= l_{fit} -L' + l_t + l_{\omega} + l_A + l_{\Omega} + l_n + l_G + l_{\phi},
\end{align}

Ahora, se definen los parámetros de influencia en la longitud medida y que se reconocen en la ecuación arriba mencionada como sigue:

- $l_{fit}$ es el valor de longitud encontrado por medio del método de fracciones excedentes.
- La corrección por temperatura está dada por la temperatura del bloque patrón ($t_g$), el coeficiente de dilatación termica del material del bloque patrón ($\alpha$). Se expresa como:

\begin{equation}
    l_t = \theta \alpha L,
\end{equation}

donde $\theta$ es el desface de temperatura entendido como $\theta = 20 - t_g$.

In [2]:
# gauge temperature correction
def gauge_temperature_correction(t_g, alpha, L): return (20 - t_g) * alpha * L

- $l_{\omega}$ es la correción atribuida a el espesor de la capa de adherencia. El valor medio de esta correción es cero $<l_{\omega}> = 0 $, debido a que la longitud del bloque contiene este espesor en su definición. Esto por otra parte, no indica que su incertidumbre asociada $u(\omega)$ con la variacion de la capa de adherencia sea cero.
- La correción por errores en el frente de onda resultantes de imperfecciones en las opticas es $l_A$, con una incertumbre asociada $u(l_A)$. El valor esperado de esta correción es cero $<l_A> = 0$, más su incertidumbre es diferente de cero.
- La correción por oblicuidad es una corrección en la longitud que tiene en cuenta el cambio de fase resultante del diseño óptico y alineación inherentes al montaje interferometrico,

\begin{equation}
    l_{\Omega} = \Omega L' = \bigg(\frac{a^2}{16f^2} + \frac{x^2}{2f^2} \bigg)L'
\end{equation}

donde $f$ es la distancia focal del colimador, $a$ es el diametro de la apertura y $x$ es el desplazamiento lateral. 

In [3]:
def obliquity_correction(a: float, f: float, x: float, L: int): return (a**2/(16 * f**2) + (x**2/(2 * f**2))) * L 

- The refractive index correction is evaluated using a modified version of the Edlén equation:

\begin{equation}
    l_n = (n - 1)L'
\end{equation}

donde $n$ es el índice de refracción del aire evaluado usado una versión de la ecuación de Edlen.

In [4]:
def refractive_index_correction(refractive_index_air: float, L: float) -> float: return (refractive_index_air - 1) * L

- La corrección por geometría del bloque patrón $l_G$ tiene en cuenta la falta de planitud y paralelismo del bloque patrón. La incertudmbre asociadad con esta correción es $u(l_G)$.
- La correción por cambio de fase tiene en cuenta la diferencia aparente entre la longitud de camino óptico y la longitud de camino mecanico. Esta correción compensa la diferencia en la forma en que la luz se refleja en diferentes materiales y superficies. Su incertidumbre es $u(l_{\phi})$. Se calcula como:

\begin{equation}
    l_{\phi} = \frac{1}{n - 1}\bigg(l_{o,p} - \sum_{i=1}^n l_{o,i}\bigg)
\end{equation}

donde $l_{o,p}$ y $l_{o,i}$ son las longitudes ópticas del paquete de bloques y de cada uno de los $i$ bloques individuales. Estas longitudes no deben confundirse con las longitudes mecánicas. Puesto que las últimas ya contienen el termino de correción por cambio de fase. En otras palabras, la longitud óptica es una medida incompleta, es el resultado obtenido por algun método de analisis de franjas (excedentes fraccionarios, phase-stepping, etc.) sin tener en cuenta la correción por cambio de fase. Este valor una vez encontrado se puede agregar a cualquier medición futura de bloques de es mismo material y acabado.

In [ ]:
def change_phase_correction(n: float, optical_length_pack: float, optical_individual_length: tuple[float]) -> float:
    return (1 / ( n - 1 )) * (optical_length_pack - np.sum(optical_individual_length))
    